<a href="https://colab.research.google.com/github/KayTheCoder-101/urdu-question-generator-/blob/main/urdu_question_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment

In [1]:
pip install datasets sentencepiece sacrebleu rouge-score torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=cc889fa93277849f2fd3e5e2404bba834ca4094393efba0237b91f1ba3f9185c
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


## Load the dataset and inspect it

In [2]:
from datasets import load_dataset

ds = load_dataset("uqa/UQA")
print ( ds )


README.md:   0%|          | 0.00/898 [00:00<?, ?B/s]

data/train-00000-of-00001-bac007e8ca7192(…): reconstructing file:   0%|          |  0.00B / 30.2MB            

data/train-00000-of-00001-bac007e8ca7192(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-cf8a6960d(…): reconstructing file:   0%|          |  0.00B / 2.92MB            

data/validation-00000-of-00001-cf8a6960d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/124745 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/16824 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 124745
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 16824
    })
})


In [3]:
ex=ds["train"][0]
print(ex.keys())

dict_keys(['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'])


In [4]:
print(ex)

{'id': '56be85543aeaaa14008c9063', 'title': 'بیونسے', 'context': "Beyoncé Giselle Knowles-Carter (/biː'j ⁇ nseɪ/ bee-YON-say) (پیدائش 4 ستمبر 1981) ایک امریکی گلوکارہ ، گانا لکھنے والی ، ریکارڈ پروڈیوسر اور اداکارہ ہیں۔ ہیوسٹن ، ٹیکساس میں پیدا ہوئی اور اس کی پرورش ہوئی ، اس نے بچپن میں مختلف گانے اور رقص کے مقابلوں میں پرفارم کیا ، اور 1990 کی دہائی کے آخر میں R&B گرل گروپ ڈسٹنی چائلڈ کے لیڈ گلوکار کی حیثیت سے شہرت حاصل کی۔ اس کے والد ، میتھیو نولز کے زیر انتظام ، یہ گروپ دنیا کے سب سے زیادہ فروخت ہونے والے گرل گروپس میں سے ایک بن گیا۔ ان کے وقفے نے بیونس کی پہلی البم ، خطرناک طور پر محبت میں (2003) کی رہائی دیکھی ، جس نے اسے دنیا بھر میں ایک سولو آرٹسٹ کے طور پر قائم کیا ، پانچ گریمی ایوارڈز حاصل کیے اور بل بورڈ ہاٹ 100 میں نمبر ون سنگلز میں محبت اور بچے پاگل لڑکے۔", 'question': 'بیونس نے کب مقبولیت حاصل کرنا شروع کی؟', 'is_impossible': False, 'answer': '1990 کی دہائی کے آخر میں', 'answer_start': 273}


In [5]:
print(ex["question"])

بیونس نے کب مقبولیت حاصل کرنا شروع کی؟


In [6]:
print(ex["answer"])

1990 کی دہائی کے آخر میں


In [7]:
n_total = len(ds["train"])
n_ans = sum(len(ex["answer"]) > 0 for ex in ds["train"])
print(f"train rows: {n_total}, answerable: {n_ans}")

train rows: 124745, answerable: 83018


In [8]:
n_total = len(ds["train"])
n_ans = sum(not ex["is_impossible"] for ex in ds["train"])
print(f"train rows: {n_total}, answerable: {n_ans}")

train rows: 124745, answerable: 83018


# Build sentence-level source/target pairs

In [9]:
import csv

In [14]:
ANS_OPEN, ANS_CLOSE= "<ans>", "</ans>"
SENT_DELIMS = "\u06D4\u061F!"

def split_sentences(text):
    start = 0
    for i, ch in enumerate(text):
        if ch in SENT_DELIMS:
            yield start, i + 1, text[start:i + 1]
            start = i + 1
    if start < len(text):
        yield start, len(text), text[start:]

def make_pair(example, max_src=60, max_tgt=25):
    """Return (source, target) or None if the row is unusable."""
    a_text = example["answer"]
    if not a_text:  # empty/unanswerable -> skip
        return None

    context = example["context"]
    a_start = context.find(a_text)
    if a_start == -1:
        return None  # answer text not found verbatim in context -> skip

    a_end = a_start + len(a_text)

    for s, e, sent in split_sentences(context):
        if s <= a_start < e:
            rel = a_start - s  # offset inside the sentence
            if sent[rel:rel + len(a_text)] != a_text:
                return None  # offset mismatch -> skip

            src = (sent[:rel] + " " + ANS_OPEN + " " + a_text + " "
                   + ANS_CLOSE + " " + sent[rel + len(a_text):]).strip()
            src = " ".join(src.split())  # normalise whitespace
            tgt = " ".join(example["question"].split())

            if len(src.split()) > max_src or len(tgt.split()) > max_tgt:
                return None

            return src, tgt

    return None


def build_split(split, out_path):
    pairs = [p for p in map(make_pair, split) if p is not None]
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        w.writerows(pairs)
    print(f"{out_path}: {len(pairs)} pairs")
    return pairs


train_pairs = build_split(ds["train"], "train.tsv")
valid_pairs = build_split(ds["validation"], "valid.tsv")


train.tsv: 75003 pairs
valid.tsv: 9999 pairs
